## Gamified Home Pilot (MVP-A)

Phase 1 scaffold is active. Next step is moving zone UI and game panel logic from index into UI_RUNNER and GAME_RUNNER cells.


## Zone Launcher (MVP-A)

Pick a course zone on the left, then run the game on the right. This keeps one active game panel while changing mission context.

In [ ]:
%%html
<!-- UI_RUNNER: Course zone control panel | panel: home_zone_launcher, slot: left, layout: row, ratio: 38-62, gap: 1rem -->
<div id="home-zone-control" class="runner-side-panel">
  <h3>Course Zones</h3>
  <p>Select a zone, then click Start on the game panel.</p>
  <div style="display:flex; gap:0.5rem; flex-wrap:wrap; margin:0.5rem 0 0.75rem;">
    <button class="zone-btn" data-zone="CSSE">CSSE</button>
    <button class="zone-btn" data-zone="CSP">CSP</button>
    <button class="zone-btn" data-zone="CSA">CSA</button>
  </div>
  <p id="home-zone-active"><strong>Active Zone:</strong> CSSE</p>
  <p id="home-zone-mission">Mission: Build confidence with game objects and interactions.</p>
  <p id="home-zone-points">Zone Points: 0</p>
  <div id="home-zone-last"></div>
</div>
<script>
window.ocsActiveZone = window.ocsActiveZone || 'CSSE';
window.ocsZonePoints = window.ocsZonePoints || { CSSE: 0, CSP: 0, CSA: 0 };

const zoneMission = {
  CSSE: 'Mission: Build confidence with game objects and interactions.',
  CSP: 'Mission: Explore algorithms, data, and procedural thinking.',
  CSA: 'Mission: Strengthen object behavior, methods, and abstraction.'
};

const renderZonePanel = () => {
  const zone = window.ocsActiveZone || 'CSSE';
  const label = document.getElementById('home-zone-active');
  const mission = document.getElementById('home-zone-mission');
  const points = document.getElementById('home-zone-points');

  if (label) label.innerHTML = '<strong>Active Zone:</strong> ' + zone;
  if (mission) mission.textContent = zoneMission[zone] || zoneMission.CSSE;
  if (points) points.textContent = 'Zone Points: ' + (window.ocsZonePoints[zone] || 0);
};

document.querySelectorAll('#home-zone-control .zone-btn').forEach((btn) => {
  btn.addEventListener('click', () => {
    window.ocsActiveZone = btn.dataset.zone;
    renderZonePanel();
  });
});

renderZonePanel();
</script>

In [ ]:
%%js

// GAME_RUNNER: Zone challenge runner | hide_edit: true, panel: home_zone_launcher, slot: right, layout: row, ratio: 38-62, gap: 1rem, width: 100%, height: 520px
import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import Clicker from '/assets/js/GameEnginev1.1/essentials/Clicker.js'

class HomeZoneGame {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;
    const zone = window.ocsActiveZone || 'CSSE';

    const cookie_src = path + '/hacks/cookie-clicker/assets/baseCookie.png';
    const grandma_src = path + '/hacks/cookie-clicker/assets/grandma.png';

    const zoneConfig = {
      CSSE: { speed: [1, 4], count: 2 },
      CSP: { speed: [2, 6], count: 3 },
      CSA: { speed: [3, 8], count: 4 }
    };
    const cfg = zoneConfig[zone] || zoneConfig.CSSE;

    const spriteData = {
      id: zone + ' Target',
      greeting: 'Click me to score in ' + zone + '!',
      src: cookie_src,
      SCALE_FACTOR: 8,
      pixels: { height: 512, width: 512 },
      INIT_POSITION: { x: 0, y: 0 },
      orientation: { rows: 1, columns: 1 },
      down: { row: 0, start: 0, columns: 1, wiggle: 0.10 },
      up: { row: 0, start: 0, columns: 1, wiggle: 0.10 },
      left: { row: 0, start: 0, columns: 1, wiggle: 0.10 },
      right: { row: 0, start: 0, columns: 1, wiggle: 0.10 },
      hitbox: { widthPercentage: 0.15, heightPercentage: 0.15 },
      walkingArea: { xMin: 0, xMax: width, yMin: 0, yMax: height },
      speed: 3,
      direction: { x: 1, y: 1 },
      interact: function(clicks, objectId) {
        const currentZone = window.ocsActiveZone || zone;
        const points = window.ocsZonePoints || (window.ocsZonePoints = { CSSE: 0, CSP: 0, CSA: 0 });
        points[currentZone] = (points[currentZone] || 0) + 1;

        const pointsEl = document.getElementById('home-zone-points');
        if (pointsEl) pointsEl.textContent = 'Zone Points: ' + points[currentZone];

        const lastEl = document.getElementById('home-zone-last');
        if (lastEl) {
          const id = objectId || (this && this.spriteData && this.spriteData.id) || 'target';
          lastEl.innerHTML = '<strong>Last Action:</strong> clicked ' + id + ', local clicks=' + clicks;
        }

        const metric = document.querySelector('[id^="gamerunner-home-gamified-mvp"][id$="-metric"]');
        if (metric) metric.textContent = currentZone + ' Points: ' + points[currentZone];
      },
    };

    this.classes = [{
      class: Clicker,
      data: spriteData,
      spawn: {
        count: cfg.count,
        ranges: {
          INIT_POSITION: {
            x: [0, Math.max(0, width - 128)],
            y: [0, Math.max(0, height - 128)]
          },
          speed: cfg.speed
        },
        pickOne: { src: [cookie_src, grandma_src] }
      }
    }];
  }
}

export const gameLevelClasses = [HomeZoneGame];
export { GameControl };

Former Home Page: [Mario Gamified Navigation]({{ site.baseurl }}/home2)